#  Fake News Detection System

## 1. Environment Setup

In [27]:

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
import os

print("Libraries imported successfully!")


Libraries imported successfully!


## 2. Load and Prepare LIAR Dataset
The LIAR dataset is more complex than ISOT, labeled with 6 degrees of truth. We will map them to Binary (True/Fake) for compatibility.

In [2]:
def load_liar_dataset(path):
    # LIAR columns: ID, label, statement, subjects, speaker, ...
    # We only need label and statement usually.
    col_names = ['id', 'label', 'statement', 'subject', 'speaker', 'job', 'state', 'party', 'barely_true_c', 'false_c', 'half_true_c', 'mostly_true_c', 'pants_on_fire_c', 'context']
    try:
        df = pd.read_csv(path, sep='\t', header=None, names=col_names, quoting=3) # quoting=3 (QUOTE_NONE) helps with parsing errors
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return pd.DataFrame() # Return empty if fails
    return df[['label', 'statement']]

def map_liar_labels(label):
    # Mapping 6 classes to 2 (Fake=1, Real=0)
    if label in ['pants-fire', 'false', 'barely-true']:
        return 1 # Fake
    elif label in ['true', 'mostly-true']:
        return 0 # Real
    else:
        return -1 # Ambiguous (half-true)

# Load Data
try:
    train_df = load_liar_dataset('../Data/LiarData/train.tsv')
    test_df = load_liar_dataset('../Data/LiarData/test.tsv')
    valid_df = load_liar_dataset('../Data/LiarData/valid.tsv')

    # Combine for full processing
    liar_df = pd.concat([train_df, test_df, valid_df])
    
    if not liar_df.empty:
        # Apply mapping
        liar_df['binary_label'] = liar_df['label'].apply(map_liar_labels)
        
        # Remove ambiguous
        liar_df = liar_df[liar_df['binary_label'] != -1].reset_index(drop=True)
        
        print(f"LIAR Dataset Loaded: {len(liar_df)} rows")
        print(liar_df['binary_label'].value_counts())
    else:
        print("LIAR Dataset loaded but empty or failed.")
    
except FileNotFoundError:
    print("LIAR dataset not found. Please ensure files are in ../Data/LiarData/")

LIAR Dataset Loaded: 10198 rows
binary_label
1    5669
0    4529
Name: count, dtype: int64


## 3. Prepare ISOT Dataset (for Training)
We will train on ISOT (large, clean) and test on LIAR (complex, unseen) to check robustness.

In [3]:
# Load ISOT
fake_df = pd.read_csv('../Data/Fake.csv')
true_df = pd.read_csv('../Data/True.csv')

fake_df['label'] = 1
true_df['label'] = 0
import re
def clean_text(text):

    text = re.sub(r"^.*?\(Reuters\)\s*-\s*", "", text) 
    text = re.sub(r"^WASHINGTON\s*-\s*", "", text)
    return text
# Apply cleaning
true_df['text'] = true_df['text'].apply(clean_text)
fake_df['text'] = fake_df['text'].apply(clean_text)

isot_df = pd.concat([fake_df, true_df]).sample(frac=1, random_state=42).reset_index(drop=True)

# Combine Text + Title for better context
isot_df['text_combined'] = isot_df['title'] + " " + isot_df['text']

# Sampling for Speed (Optional: Remove .head(1000) for full training)
# IMPORTANT: Fine-tuning BERT on CPU is slow. We use a subset for demonstration
isot_subset = isot_df.head(2000) # 2000 samples for demo

print(f"Training Subset Size: {len(isot_subset)}")

Training Subset Size: 2000


## 4. Tokenization & Dataset Object
Transformers require specific tokenization.

In [4]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Split Data
X_train, X_val, y_train, y_val = train_test_split(
    isot_subset['text_combined'].tolist(), 
    isot_subset['label'].tolist(), 
    test_size=0.2, 
    random_state=42
)

# Tokenize
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(X_val, truncation=True, padding=True, max_length=128)

# Create Datasets
train_dataset = NewsDataset(train_encodings, y_train)
val_dataset = NewsDataset(val_encodings, y_val)

## 5. Fine-Tuning DistilBERT
We use the `Trainer` API from Hugging Face.

In [5]:
# Load Model
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')

# Training Arguments
training_args = TrainingArguments(
    output_dir='../Models/BertResults',
    num_train_epochs=1,              # 1 Epoch for demo speed
    per_device_train_batch_size=8,   # Small batch for CPU
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir='../Models/BertLogs',
    logging_steps=10,
    eval_strategy="steps",           # UPDATED: 'evaluation_strategy' is deprecated
    eval_steps=50,
    save_steps=100,
    load_best_model_at_end=True
)

# Metrics function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {'accuracy': acc, 'f1': f1}

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Starting Training...")
try:
    trainer.train()
    print("Training Complete!")
except Exception as e:
    print(f"Training interrupted: {e}")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting Training...


Step,Training Loss,Validation Loss,Accuracy,F1
50,0.244700,0.446787,0.845000,0.876000
100,0.324000,0.184683,0.950000,0.952830
150,0.008300,0.130709,0.965000,0.967890
200,0.107900,0.125803,0.965000,0.967593


Training Complete!


## 6. Deployment: Save Model
This saves the model in a standard format loadable by cloud services.

In [6]:
save_directory = "../Models/FakeNewsDistilBO"
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"Model saved to {save_directory}")

Model saved to ../Models/FakeNewsDistilBO


## 7. Inference Demo (Advanced)
Test the model on custom text.

In [7]:
def predict_news_bert(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=-1)
    pred_idx = torch.argmax(probs).item()
    
    label = "FAKE" if pred_idx == 1 else "REAL"
    confidence = probs[0][pred_idx].item()
    
    return label, confidence

# Extended Testing Examples
examples = [
    # FAKE: Extravagant claim
    "BREAKING: Scientists discover that eating coffee powder cures cancer instantly!",
    
    # REAL: Political event (General)
    "The President visited the disaster zone today to assess the damage from the hurricane.",
    
    # FAKE: Conspiracy
    "Classified documents prove that the earth is actually flat and NASA has been lying for decades.",
    
    # REAL: Technology
    "Apple announced its new line of MacBooks featuring the M3 chip with improved battery life."
]

print("-" * 50)
print("FINAL MODEL PREDICTIONS")
print("-" * 50)

for i, text in enumerate(examples, 1):
    label, conf = predict_news_bert(text, model, tokenizer)
    print(f"\nExample {i}:")
    print(f"Text: {text}")
    print(f"Prediction: {label} (Confidence: {conf:.2%})")

--------------------------------------------------
FINAL MODEL PREDICTIONS
--------------------------------------------------

Example 1:
Text: BREAKING: Scientists discover that eating coffee powder cures cancer instantly!
Prediction: FAKE (Confidence: 99.78%)

Example 2:
Text: The President visited the disaster zone today to assess the damage from the hurricane.
Prediction: REAL (Confidence: 97.72%)

Example 3:
Text: Classified documents prove that the earth is actually flat and NASA has been lying for decades.
Prediction: FAKE (Confidence: 98.99%)

Example 4:
Text: Apple announced its new line of MacBooks featuring the M3 chip with improved battery life.
Prediction: REAL (Confidence: 70.82%)


In [8]:

import os
import shutil

print("-" * 50)
print("MODEL OPTIMIZATION (Reduce Size)")
print("-" * 50)

# 1. DELETE OPTIMIZER (Not needed for inference)
optimizer_path = os.path.join(save_directory, "optimizer.pt")
if os.path.exists(optimizer_path):
    os.remove(optimizer_path)
    print(f"Deleted 'optimizer.pt' (Saved ~500MB)")

# 2. QUANTIZATION (Float32 -> Int8)
# This makes the model 2x-4x smaller and faster on CPU
try:
    print("Applying Dynamic Quantization...")
    quantized_model = torch.quantization.quantize_dynamic(
        model, {torch.nn.Linear}, dtype=torch.qint8
    )
    
    # Save Quantized Model
    quantized_dir = "../Models/FakeNewsDistilBERT_Quantized"
    if not os.path.exists(quantized_dir):
        os.makedirs(quantized_dir)
        
    torch.save(quantized_model.state_dict(), os.path.join(quantized_dir, "pytorch_model.bin"))
    tokenizer.save_pretrained(quantized_dir)
    print(f"Quantized Model saved to {quantized_dir}")
    
    # Compare Sizes
    original_size = os.path.getsize(os.path.join(save_directory, "model.safetensors")) / (1024*1024)
    quant_size = os.path.getsize(os.path.join(quantized_dir, "pytorch_model.bin")) / (1024*1024)
    
    print(f"\nOriginal Size: {original_size:.2f} MB")
    print(f"Quantized Size: {quant_size:.2f} MB")
    print(f"Reduction: {(1 - quant_size/original_size):.2%}")

except Exception as e:
    print(f"Quantization failed: {e}")


--------------------------------------------------
MODEL OPTIMIZATION (Reduce Size)
--------------------------------------------------
Applying Dynamic Quantization...
Quantized Model saved to ../Models/FakeNewsDistilBERT_Quantized

Original Size: 255.43 MB
Quantized Size: 132.29 MB
Reduction: 48.21%
